# ShowUI-Aloha Blender Sphere → HyperData Upload

This notebook uploads the **ShowUI-Aloha** `blender_sphere` GUI-automation
dataset to HyperData for versioned storage and remote access.

| Section | Description |
|---------|-------------|
| §1 | Load screenshots, crops, action logs, and trace captions |
| §2 | Build Zarr arrays (images, crops, actions, trace) |
| §3 | Create local HyperData dataset |
| §4 | Push dataset to HyperData Hub |
| §5 | Verify by pulling back and inspecting |

## 0. Setup

```bash
uv pip install hyperdata Pillow numpy
```

The cell below sets default environment variables for the shared HyperData
deployment. Override them before running if your server differs.

In [ ]:
import os

# ---- Default HyperData remote configuration ----
os.environ.setdefault("HYPERDATA_ENDPOINT", "http://118.180.19.234:8021")
os.environ.setdefault("MINIO_ENDPOINT", "118.180.19.234")
os.environ.setdefault("MINIO_PORT", "9010")
os.environ.setdefault("MINIO_ACCESS_KEY", "hyperdata_admin")
os.environ.setdefault("MINIO_SECRET_KEY", "change-this-password-in-production")
os.environ.setdefault("S3_ENDPOINT", "118.180.19.234")
os.environ.setdefault("S3_PORT", "9010")
os.environ.setdefault("S3_ACCESS_KEY", "hyperdata_admin")
os.environ.setdefault("S3_SECRET_KEY", "change-this-password-in-production")
os.environ.setdefault("S3_BUCKET", "hyperdata-data")

import json
import re
from pathlib import Path

import numpy as np
from PIL import Image
from hyperdata import HyperData

DATA_ROOT = Path("/data/codes/ShowUI-Aloha/inputs/blender_sphere")
HYPERDATA_DIR = "/tmp/showui_blender_sphere_hd"
REMOTE_S3_URL = "s3://hyperdata-data/showui/blender_sphere.zarr"

print(f"Data root: {DATA_ROOT}")
print(f"Endpoint: {os.environ['HYPERDATA_ENDPOINT']}")

## 1. Load & Inspect ShowUI-Aloha Data

The `blender_sphere` directory contains:
- `screenshots/*.jpg` — full-screen frames (1920×1080)
- `screenshots/*.crop.jpg` — 256×256 cropped regions of interest
- `inputs/blender_sphere.txt` — raw keyboard/mouse event log
- `blender_sphere_processed_log.json` — structured action list
- `blender_sphere_trace.json` — trajectory with caption annotations

In [ ]:
screenshot_dir = DATA_ROOT / "screenshots"
input_dir = DATA_ROOT / "inputs"

full_shots = sorted([f for f in screenshot_dir.iterdir() if f.suffix == ".jpg" and ".crop." not in f.name])
crop_shots = sorted([f for f in screenshot_dir.iterdir() if f.suffix == ".jpg" and ".crop." in f.name])

print(f"Full screenshots: {len(full_shots)}")
print(f"Crop screenshots: {len(crop_shots)}")

# Peek at dimensions
sample_full = Image.open(full_shots[0])
sample_crop = Image.open(crop_shots[0])
print(f"Full size: {sample_full.size}, mode={sample_full.mode}")
print(f"Crop size: {sample_crop.size}, mode={sample_crop.mode}")

# Note: some .crop.jpg files are full-sized; we normalize later.
odd_crops = sum(1 for p in crop_shots if Image.open(p).size != (256, 256))
print(f"Crops that are not 256×256: {odd_crops}/{len(crop_shots)}")

# Load processed action log
with open(DATA_ROOT / "blender_sphere_processed_log.json") as f:
    action_log = json.load(f)
print(f"Action log entries: {len(action_log)}")
print(f"Keys per entry: {list(action_log[0].keys())}")

# Load trace captions (fix malformed JSON — missing comma after step_idx)
with open(DATA_ROOT / "blender_sphere_trace.json") as f:
    trace_raw = f.read()
trace_fixed = re.sub(r'("step_idx":\s*\d+)(\n\s*")', r'\1,\2', trace_raw)
trace_data = json.loads(trace_fixed)
trajectory = trace_data["trajectory"]
print(f"Trace entries: {len(trajectory)}")
print(f"Keys per entry: {list(trajectory[0].keys())}")

## 2. Build Zarr Arrays

We stack images into NumPy arrays and prepare structured metadata for
actions and trace captions. Everything is then written into a single
HyperData Zarr store.

In [ ]:
CROP_SIZE = (256, 256)

# Stack full screenshots: (N, H, W, 3)
full_arrays = [np.array(Image.open(p)) for p in full_shots]
full_images = np.stack(full_arrays, axis=0)  # uint8

# Some .crop.jpg files are actually full-sized; normalize all to 256×256
# so the array stacks cleanly.
crop_arrays = []
for p in crop_shots:
    img = Image.open(p)
    if img.size != CROP_SIZE:
        img = img.resize(CROP_SIZE, Image.Resampling.LANCZOS)
    crop_arrays.append(np.array(img))
crop_images = np.stack(crop_arrays, axis=0)  # uint8

print(f"full_images shape: {full_images.shape}, dtype={full_images.dtype}")
print(f"crop_images shape: {crop_images.shape}, dtype={crop_images.dtype}")

# Extract filenames as timestamps for alignment
full_names = [p.stem for p in full_shots]
crop_names = [p.stem for p in crop_shots]
print(f"\nFirst 3 full names: {full_names[:3]}")
print(f"First 3 crop names: {crop_names[:3]}")

In [ ]:
# Prepare metadata arrays (numpy string dtype works with Zarr v3)
action_json = np.array([json.dumps(entry) for entry in action_log])
trace_json = np.array([json.dumps(entry) for entry in trajectory])

# Screenshot filenames for reference
full_names_arr = np.array([p.stem for p in full_shots])
crop_names_arr = np.array([p.stem for p in crop_shots])

print(f"action_json shape: {action_json.shape}, dtype: {action_json.dtype}")
print(f"trace_json shape: {trace_json.shape}, dtype: {trace_json.dtype}")
print(f"full_names_arr shape: {full_names_arr.shape}, dtype: {full_names_arr.dtype}")

## 3. Create Local HyperData Dataset

`HyperData` wraps an IceChunk-backed Zarr store with built-in version
control. We write the arrays directly via bracket notation.

In [ ]:
import shutil

# Clean up any previous local copy
shutil.rmtree(HYPERDATA_DIR, ignore_errors=True)

ds = HyperData(HYPERDATA_DIR)

ds["full_images"] = full_images
ds["crop_images"] = crop_images
ds["action_log"] = action_json
ds["trace_captions"] = trace_json
ds["full_names"] = full_names_arr
ds["crop_names"] = crop_names_arr

# Store dataset-level metadata
ds.metadata.set("dataset", {
    "name": "showui_blender_sphere",
    "source": "ShowUI-Aloha/inputs/blender_sphere",
    "task": "Blender mesh editing (subdivide → sphere → shade smooth)",
    "num_full_screenshots": len(full_shots),
    "num_crop_screenshots": len(crop_shots),
    "num_action_entries": len(action_log),
    "num_trace_entries": len(trajectory),
    "full_resolution": list(full_images.shape[1:3]),
    "crop_resolution": [256, 256],
    "note": "Some source .crop.jpg files were full-sized; all normalized to 256×256 via LANCZOS.",
})

print(f"HyperData arrays: {ds.keys()}")
print(f"full_images: {ds['full_images'].shape}")
print(f"crop_images: {ds['crop_images'].shape}")

## 4. Push to HyperData Hub

`push_to_hub()` performs an S3 delta sync and registers the dataset in the
catalogue so it appears in `hd overview`.

In [ ]:
# Add remote and push
ds.add_remote("origin", REMOTE_S3_URL)
ds.push_to_hub("@devin/showui-blender-sphere")

print("Dataset pushed to HyperData Hub.")
print("View it with: hd overview")

## 5. Verify — Pull Back & Inspect

Confirm the remote dataset can be pulled and the data is intact.

In [ ]:
VERIFY_DIR = "/tmp/showui_blender_sphere_verify"
shutil.rmtree(VERIFY_DIR, ignore_errors=True)

ds2 = HyperData(VERIFY_DIR)
ds2.add_remote("origin", REMOTE_S3_URL)
ds2.pull("origin")

print(f"Pulled arrays: {ds2.keys()}")
print(f"full_images shape: {ds2['full_images'].shape}")
print(f"crop_images shape: {ds2['crop_images'].shape}")

# Verify first image matches
pulled_first = ds2["full_images"][0].to_numpy()
local_first = full_images[0]
match = np.array_equal(pulled_first, local_first)
print(f"First image pixel-perfect match: {match}")

# Verify a trace caption round-trips
trace_back = json.loads(ds2["trace_captions"][0].to_numpy().item())
print(f"First trace step_idx: {trace_back['step_idx']}")

# Verify metadata
meta = ds2.metadata.get("dataset")
print(f"Dataset metadata: {meta}")

## 6. (Optional) PyTorch Dataset Adapter

If you want to consume this data with Lumen's trainers, you can wrap the
HyperData arrays in a standard `torch.utils.data.Dataset`. Below is a
minimal example.

In [ ]:
import torch
from torch.utils.data import Dataset

class BlenderSphereDataset(Dataset):
    """Simple dataset over the pulled HyperData store."""
    def __init__(self, hyperdata):
        self.ds = hyperdata
        self.n = self.ds["full_images"].shape[0]

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        # Images come back as (H, W, C) uint8 → convert to (C, H, W) float32
        img = self.ds["full_images"][idx].to_numpy().astype(np.float32) / 255.0
        img = torch.from_numpy(img).permute(2, 0, 1)
        crop = self.ds["crop_images"][idx].to_numpy().astype(np.float32) / 255.0
        crop = torch.from_numpy(crop).permute(2, 0, 1)
        return {
            "image": img,
            "crop": crop,
            "image_name": self.ds["full_names"][idx].to_numpy().item(),
            "crop_name": self.ds["crop_names"][idx].to_numpy().item(),
        }

dataset = BlenderSphereDataset(ds2)
sample = dataset[0]
print(f"Sample keys: {sample.keys()}")
print(f"image shape: {sample['image'].shape}, dtype={sample['image'].dtype}")
print(f"crop shape: {sample['crop'].shape}, dtype={sample['crop'].dtype}")
print(f"name: {sample['image_name']}")

## Cleanup

In [ ]:
shutil.rmtree(HYPERDATA_DIR, ignore_errors=True)
shutil.rmtree(VERIFY_DIR, ignore_errors=True)
print("Cleaned up temp directories.")